In [1]:
import sys
sys.path.append('..')

import numpy as np
import json
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score

from models.decision_tree import DecisionTreeModel

# Configs
json_path = '../models/decision_tree.json'
random_seed = 42

# Randomly generated sample data
np.random.seed(random_seed)
X, y = make_classification(
    n_samples=300,
    n_features=4,
    n_informative=3,
    n_redundant=1,
    n_classes=3,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")
print(f"Features: {X.shape[1]}")
print(f"Classes: {len(np.unique(y))}")

Training samples: 240
Testing samples: 60
Features: 4
Classes: 3


In [2]:
sklearn_decision_tree = DecisionTreeClassifier(
    max_depth=5,
    random_state=random_seed
)
sklearn_decision_tree.fit(X_train, y_train)

y_pred = sklearn_decision_tree.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Tree depth: {sklearn_decision_tree.get_depth()}")
print(f"Number of leaves: {sklearn_decision_tree.get_n_leaves()}")
print(f"Number of nodes: {sklearn_decision_tree.tree_.node_count}")

Accuracy: 0.8333
Tree depth: 5
Number of leaves: 22
Number of nodes: 43


In [3]:
def extract_tree_to_dict(tree, feature_names=None):
    tree_ = tree.tree_
    
    if feature_names is None:
        feature_names = [f"feature_{i}" for i in range(tree_.n_features)]
    
    def build_json(node_id, depth):
        left_child = tree_.children_left[node_id]
        right_child = tree_.children_right[node_id]
        feature = tree_.feature[node_id]
        threshold = tree_.threshold[node_id]
        value = tree_.value[node_id]
        n_samples = tree_.n_node_samples[node_id]
        impurity = tree_.impurity[node_id]
        
        is_leaf = (left_child == right_child)
        
        node_dict = {
            'node_id': int(node_id),
            'depth': int(depth),
            'n_samples': int(n_samples),
            'impurity': float(impurity),
            'is_leaf': bool(is_leaf)
        }
        
        if is_leaf:
            class_counts = value[0].tolist()
            predicted_class = int(np.argmax(value[0]))
            node_dict.update({
                'class_distribution': class_counts,
                'predicted_class': predicted_class
            })
        else:
            node_dict.update({
                'feature': int(feature),
                'feature_name': feature_names[feature],
                'threshold': float(threshold),
                'left': build_json(left_child, depth + 1),
                'right': build_json(right_child, depth + 1)
            })
        
        return node_dict
    
    tree_structure = build_json(0, 0)
    
    tree_dict = {
        'metadata': {
            'n_features': int(tree_.n_features),
            'n_classes': int(tree_.n_classes[0]),
            'n_outputs': int(tree_.n_outputs),
            'max_depth': int(tree_.max_depth),
            'node_count': int(tree_.node_count),
            'feature_names': feature_names
        },
        'tree': tree_structure
    }
    
    return tree_dict

In [4]:
tree_dict = extract_tree_to_dict(sklearn_decision_tree)

with open(json_path, 'w') as f:
    json.dump(tree_dict, f, indent=2)

print(f"Tree saved to: {json_path}")

Tree saved to: ../models/decision_tree.json


In [5]:
custom_tree = DecisionTreeModel(json_path)

Tree loaded from: ../models/decision_tree.json


In [6]:
sklearn_predictions = sklearn_decision_tree.predict(X_test)
custom_predictions = custom_tree.predict(X_test)

matches = np.sum(sklearn_predictions == custom_predictions)
total = len(X_test)

print(f"Predictions match: {matches}/{total}")
print(f"\nsklearn accuracy: {accuracy_score(y_test, sklearn_predictions):.4f}")
print(f"Custom accuracy: {accuracy_score(y_test, custom_predictions):.4f}")

print("\nFirst 10 predictions:")
print("Index | sklearn | Custom | Actual | Match")
print("-" * 50)
for i in range(min(10, len(X_test))):
    sk_pred = sklearn_predictions[i]
    cu_pred = custom_predictions[i]
    actual = y_test[i]
    match = "True" if sk_pred == cu_pred else "False"
    print(f"  {i:2d}  |    {sk_pred}    |   {cu_pred}    |   {actual}    |  {match}")

Predictions match: 60/60

sklearn accuracy: 0.8333
Custom accuracy: 0.8333

First 10 predictions:
Index | sklearn | Custom | Actual | Match
--------------------------------------------------
   0  |    2    |   2    |   0    |  True
   1  |    0    |   0    |   0    |  True
   2  |    2    |   2    |   2    |  True
   3  |    2    |   2    |   2    |  True
   4  |    2    |   2    |   2    |  True
   5  |    0    |   0    |   0    |  True
   6  |    1    |   1    |   1    |  True
   7  |    0    |   0    |   0    |  True
   8  |    0    |   0    |   2    |  True
   9  |    0    |   0    |   0    |  True


In [7]:
test_sample = X_test[0]

print("Single input prediction:")
print(f"Features: {test_sample}")
print(f"sklearn prediction: {sklearn_decision_tree.predict([test_sample])[0]}")
print(f"Custom prediction: {custom_tree.predict(test_sample)[0]}")
print(f"Actual class: {y_test[0]}")

Single input prediction:
Features: [-0.35496286 -0.67115921 -0.32520062  0.87207138]
sklearn prediction: 2
Custom prediction: 2
Actual class: 0
